# Uncertainty-Aware Polyp Segmentation on Kvasir-SEG

**Student:** Rio Roy  
**Module:** Computer Vision and Artificial Intelligence  
**Experiment:** A controlled comparison of a compact U-Net and Attention U-Net, followed by Monte Carlo dropout uncertainty analysis.

This notebook is the reproducible companion to the written report. It validates the real Kvasir-SEG image-mask pairs, loads the deterministic split and executed results, and exposes the implementation used to train and evaluate both models.

## Context and methods

- Data: 1,000 expert-annotated colonoscopy images and paired masks from Kvasir-SEG.
- Split: 800 training, 100 validation and 100 test images using seed 42.
- Input: RGB images resized to 96 x 96 and scaled to [0, 1]; masks resized with nearest-neighbour interpolation and thresholded at 127.
- Training: 10 epochs, batch size 16, Adam (learning rate 0.001), BCE plus soft Dice loss.
- Comparison: identical split, resolution, optimiser and evaluation threshold for both models.
- Original extension: dropout-enabled Attention U-Net with 12 stochastic inference passes to visualise epistemic uncertainty.

The reduced input resolution and compact base width make the experiment reproducible on CPU. They also limit boundary detail and mean the results should not be interpreted as clinical performance.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT = Path.cwd()
if not (PROJECT / 'train_models.py').exists():
    PROJECT = Path.cwd() / 'cvai_project'
ROOT = PROJECT.parent
DATA = ROOT / 'Kvasir-SEG' / 'Kvasir-SEG'
OUTPUTS = PROJECT / 'outputs'
sys.path.insert(0, str(PROJECT))

from train_models import UNet, KvasirDataset, metrics_from_counts

print('Project:', PROJECT.resolve())
print('Dataset:', DATA.resolve())
print('Results:', OUTPUTS.resolve())

## Data validation

In [ ]:
image_paths = sorted((DATA / 'images').glob('*.jpg'))
mask_paths = sorted((DATA / 'masks').glob('*.jpg'))
image_names = {p.name for p in image_paths}
mask_names = {p.name for p in mask_paths}

assert len(image_paths) == 1000
assert len(mask_paths) == 1000
assert image_names == mask_names

split = json.loads((OUTPUTS / 'split.json').read_text())
assert [len(split[k]) for k in ['train', 'validation', 'test']] == [800, 100, 100]
assert len(set(split['train']) | set(split['validation']) | set(split['test'])) == 1000

print({'images': len(image_paths), 'masks': len(mask_paths),
       'train': len(split['train']), 'validation': len(split['validation']),
       'test': len(split['test']), 'overlap': 0})

In [ ]:
sample_names = split['train'][:3]
fig, axes = plt.subplots(3, 2, figsize=(7, 9))
for row, name in enumerate(sample_names):
    image = Image.open(DATA / 'images' / name).convert('RGB')
    mask = Image.open(DATA / 'masks' / name).convert('L')
    axes[row, 0].imshow(image)
    axes[row, 1].imshow(mask, cmap='gray')
    axes[row, 0].set_title(f'Image: {name[:12]}...')
    axes[row, 1].set_title('Expert mask')
    for ax in axes[row]: ax.axis('off')
plt.tight_layout();

## Architecture and evaluation logic

The implementation is held in `train_models.py`. U-Net uses encoder-decoder blocks and skip concatenations. Attention U-Net applies a learned sigmoid attention gate to every skip connection. The latter also contains spatial dropout, which remains active during repeated inference for uncertainty estimation.

In [ ]:
unet = UNet(base=8, attention=False, dropout=0.0)
attention_unet = UNet(base=8, attention=True, dropout=0.20)
parameter_table = pd.DataFrame({
    'Model': ['U-Net', 'Attention U-Net'],
    'Parameters': [sum(p.numel() for p in unet.parameters()),
                   sum(p.numel() for p in attention_unet.parameters())]
})
parameter_table['Increase vs U-Net (%)'] = ((parameter_table['Parameters'] / parameter_table.loc[0, 'Parameters']) - 1) * 100
parameter_table.round(2)

In [ ]:
def segmentation_metrics_from_logits(logits, targets, threshold=0.5):
    probabilities = logits.sigmoid()
    predicted = probabilities >= threshold
    actual = targets >= 0.5
    tp = (predicted & actual).sum().item()
    fp = (predicted & ~actual).sum().item()
    fn = (~predicted & actual).sum().item()
    tn = (~predicted & ~actual).sum().item()
    return metrics_from_counts(tp, fp, fn, tn)

print('Metrics: Dice, IoU, precision, recall, specificity and pixel accuracy')

## Executed results

In [ ]:
results = json.loads((OUTPUTS / 'results.json').read_text())
rows = []
for model in ['U-Net', 'Attention U-Net']:
    row = {'Model': model, **results[model]}
    rows.append(row)
results_table = pd.DataFrame(rows)[['Model', 'dice', 'iou', 'precision', 'recall',
                                    'specificity', 'pixel_accuracy', 'parameters',
                                    'training_seconds', 'inference_ms_per_image']]
results_table.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['dice', 'iou', 'inference_ms_per_image']):
    ax.bar(results_table['Model'], results_table[metric], color=['#2457A7', '#D97706'])
    ax.set_title(metric.replace('_', ' ').title())
    ax.grid(axis='y', alpha=.25)
    if metric != 'inference_ms_per_image': ax.set_ylim(0, 1)
plt.tight_layout();

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, filename, title in zip(
    axes,
    ['learning_curves.png', 'prediction_comparison.png', 'uncertainty_example.png'],
    ['Learning curves', 'Qualitative test predictions', 'MC-dropout uncertainty']):
    ax.imshow(Image.open(OUTPUTS / filename))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout();

## Publication-strength robustness analysis

In [ ]:
publication = json.loads((PROJECT / 'publication_outputs' / 'publication_results.json').read_text())
seed_table = pd.DataFrame(publication['seed_summaries'])
seed_table[['seed', 'model', 'mean_dice', 'mean_iou', 'mean_precision', 'mean_recall']].round(4)

In [ ]:
pd.DataFrame([
    {'Model': model,
     'Mean Dice': values['dice_mean_ci95'][0],
     'Dice CI low': values['dice_mean_ci95'][1],
     'Dice CI high': values['dice_mean_ci95'][2],
     'Mean IoU': values['iou_mean_ci95'][0]}
    for model, values in publication['model_ci'].items()
]).round(4)

In [ ]:
publication['paired_wilcoxon_attention_minus_unet'], publication['uncertainty_error_spearman_seed42']

In [ ]:
pd.DataFrame(publication['risk_coverage_seed42']).round(4)

In [ ]:
plt.figure(figsize=(10, 4))
plt.imshow(Image.open(PROJECT / 'report_figures' / 'publication_robustness.png'))
plt.axis('off'); plt.tight_layout();

## Findings and limitations

Across three seeds, compact U-Net achieved higher mean per-image Dice than Attention U-Net. The paired Wilcoxon result was significant at the 5% level, but the effect was small and bootstrap intervals overlapped. This supports a limited conclusion: attention did not provide a consistent advantage under the compact protocol.

MC-dropout variance correlated weakly and non-significantly with Dice error, and referral performance was non-monotonic. The tested uncertainty score should therefore not be used as a safety gate. Further work should evaluate boundary-focused uncertainty, predictive entropy, mutual information and ensembles at higher resolution and on external datasets.

## Reproduction

Run `python train_models.py` from the project directory to reproduce training, saved weights, metrics and figures. The official Kvasir-SEG source is https://datasets.simula.no/kvasir-seg/. Dataset use is restricted to research and education, and the dataset paper must be cited.